# Week 4 — Wednesday: Joining Tables — Keys and Relationships

**DATA 202 · Calvin University**

Monday's question was *"what is one row about?"* Today we ask it of **two tables at once** — and then ask the harder one:

> **Do these two rows describe the *same thing*?**

A join is a **claim** that a row in one table and a row in another are about the same thing. `pd.merge()` can't check that claim — it only checks whether two key values are *equal*. Checking the claim is our job.

**Setup:** a biomaterials lab. Five tables, from different tests on (mostly) the same materials — some joins between them are great, some run without error and mean nothing.

**Today's plan (50 min):**

| Time | Part |
|---|---|
| ~5 min | Load the five tables and map them |
| ~10 min | **Part 1 — Keys and Relational Structure** (SLO 04A) |
| ~15 min | **Part 2 — The Four Join Types** (SLO 04B) |
| ~15 min | **Part 3 — Does This Join Make Sense?** (SLO 04B) |
| ~5 min | Careful with joining + what's next |

**Cues:** 💬 Question · 🔨 Task

---
## Loading the Tables · ~5 min

In [ ]:
import pandas as pd

BASE = "https://cs.calvin.edu/courses/data/202/fsantos/biomaterial/"

properties             = pd.read_csv(BASE + "biomaterial_properties.csv")
biocompatibility       = pd.read_csv(BASE + "biomaterial_biocompatibility.csv")
degradation            = pd.read_csv(BASE + "biomaterial_degradation.csv")
environment_conditions = pd.read_csv(BASE + "environment_conditions.csv")
biocompatibility_time  = pd.read_csv(BASE + "biocompatibility_time.csv")

tables = {
    "properties": properties,
    "biocompatibility": biocompatibility,
    "degradation": degradation,
    "environment_conditions": environment_conditions,
    "biocompatibility_time": biocompatibility_time,
}
for name, t in tables.items():
    print(f"--- {name}  {t.shape}")
    display(t)

Monday's question, asked of each table:

| Table | One row is about… | Identifying column(s) |
|---|---|---|
| `properties` | one **material** and its mechanical properties | `Material ID` |
| `biocompatibility` | one **cell test** of a material (one condition, one duration in *days*) | `Material ID` |
| `biocompatibility_time` | one material's cell viability at day 7, 14 and 28 *(a wide table!)* | `Material ID` + `Condition` |
| `degradation` | one **degradation test**: a material sitting in an environment for some *weeks* | `Material ID` + `Environment` |
| `environment_conditions` | one **environment** (its pH, temperature, composition) | `Environment` |

Sizes: 8, 8, 8, 13 and 6 rows. Five tables, **different tests, different units, different durations**.

---
## Part 1 — Keys and Relational Structure (SLO 04A) · ~10 min

To connect a row in one table to a row in another we need a column that names **which** thing, identically, in both. That column is a **key**.

Think of a library. Every book has a call number, and that call number belongs to exactly one book — no two books share it. A sign-out card doesn't copy the whole book record onto itself; it just writes down that one call number, pointing back to the shelf where the real record lives. The same two roles show up in tables:

* **Primary key** — uniquely identifies each row in *its own* table, the way a call number identifies one book. `Material ID` in `properties`; `Environment` in `environment_conditions`.
* **Foreign key** — a column that *points to* a primary key in another table, the way a sign-out card points to a call number: "this row is about that thing, over there." It doesn't uniquely identify rows in its own table — the same value can, and usually does, show up more than once.

A quick way to tell which one you're looking at: pick a column and ask *"if I listed every value in this column, would each one appear exactly once?"* Yes → primary key. No, the same value repeats, each time pointing back to one shared record elsewhere → foreign key.

---
### 🔨 Task — Draw the Relational Structure (~4 min)

Grab paper (or come to the whiteboard). Using the "Identifying column(s)" from the table above, draw the five tables as boxes and connect them:

1. For each table, which column is its **primary key** — the one that's unique, one row per thing?
2. Which tables share a column with another table? On the side where that column *repeats*, it's a **foreign key**, and it points to the table where the same column is the primary key.
3. Draw a box per table and an arrow for every foreign key → primary key relationship you find.

<details><summary>Hint</summary>

`Material ID` is unique in `properties` (one row per material) — so it's the primary key there. It also appears in `biocompatibility`, `biocompatibility_time`, and `degradation`, but repeats in each of those — so in those three tables it's a **foreign key** pointing back to `properties`.
</details>

<details><summary>Reveal the diagram</summary>

```
properties ──── Material ID ────┬──── biocompatibility
 (8 materials)                  ├──── biocompatibility_time
                                └──── degradation ──── Environment ──── environment_conditions
```

`properties` is the hub: `Material ID` is its primary key, and a foreign key in the three tables pointing back to it. `degradation` is also on the *other* end of a relationship: its `Environment` column is a foreign key into `environment_conditions`, where `Environment` is the primary key.

Compare this to what you drew — did you find all four arrows? A key only works if the values match **exactly** across both sides — `pd.merge()` compares text, not meaning.
</details>

---
### 🔨 Quick Task — Is Anything Missing? (~2 min)

`degradation` and `properties` should describe the same materials. Check: is every `Material ID` in `degradation` also found in `properties`?

<details><summary>Hint</summary>

`.isin()` gives one `True`/`False` per row; `.all()` collapses that to a single answer: `degradation["Material ID"].isin(properties["Material ID"]).all()`
</details>

<details><summary>Answer</summary>

`False` — something doesn't match. That only tells us **that** there's a mismatch, not **which** rows, or what else is different about them.
</details>

In [ ]:
# Your code here

### 🔨 Mini-Task A — Find the Orphans With `.isin()` (~3 min)

The check above told us *that* something doesn't match — but not *which* rows, or what else they carry. A missing partner row means **"no record"**, not "the material doesn't exist." To actually **inspect what is in what**, filter the real dataframes:

1. Filter `degradation` to rows whose `Material ID` is **not** in `properties["Material ID"]` → `deg_orphans`
2. Filter `properties` to rows whose `Material ID` is **not** in `degradation["Material ID"]` → `props_untested`

<details><summary>Hint</summary>

`df[~df["col"].isin(other_df["col"])]` — the `~` flips "is in" to "is not in".
</details>

<details><summary>Check yourself</summary>

`deg_orphans` has 6 rows (`B009`–`B014`); `props_untested` has 1 row (`B004`, Collagen).
</details>

In [ ]:
# Your code here

---
## Part 2 — The Four Join Types (SLO 04B) · ~15 min

`pd.merge()` walks through both tables and, for each key value, asks one question: *does this row have a partner on the other side?* Every row either finds a match or it doesn't — `how=` is just the instruction for what to do with the ones that don't:

| `how=` | Keeps | Unmatched rows get... |
|:---|:---|:---|
| `"inner"` | only rows matched in **both** tables | dropped completely, from both sides |
| `"left"` | every row from the **left** table | `NaN` filled in for the right table's columns |
| `"right"` | every row from the **right** table | `NaN` filled in for the left table's columns |
| `"outer"` | every row from **either** table | `NaN` filled in on whichever side is missing |

`NaN` here just means "no record" — pandas has nothing to fill in because that row's partner never existed on the other side.

We start with the most natural pair: what a material *is* (`properties`) and how cells *react* to it (`biocompatibility`) — both keyed on `Material ID`, and a material's mechanical properties don't depend on which cell test it got, so the two rows really do describe the same thing. One row of the result will be about one material, with one of its cell tests. Picture the two tables as overlapping sets, split by whether a `Material ID` has a partner on the other side:

> 💬 **Question:** From Part 1: **6** materials are in both tables, **2** (`B005`, `B008`) only in `properties`, **2** (`B009`, `B010`) only in `biocompatibility`. Predict the row count for `inner`, `left`, `right` and `outer`.

<details><summary>Answer</summary>

inner **6** · left **6 + 2 = 8** · right **6 + 2 = 8** · outer **6 + 2 + 2 = 10** — matches the diagram above.
</details>

In [ ]:
for how in ["inner", "left", "right", "outer"]:
    joined = pd.merge(properties, biocompatibility, on="Material ID", how=how)
    print(f"{how:6s}", joined.shape)

Same two tables, four different answers. `indicator=True` adds a `_merge` column that says where each row came from — the quickest way to see who found a partner:

In [ ]:
outer = pd.merge(properties, biocompatibility, on="Material ID", how="outer", indicator=True)
outer[["Material ID", "Material Name", "Cell Viability (%)", "_merge"]]

> 💬 **Question:** Which materials are `left_only`? `right_only`? What does the `NaN` in `Cell Viability (%)` for a `left_only` row mean — zero viability, or something else?

<details><summary>Answer</summary>

`left_only`: `B005` Titanium Alloy and `B008` Magnesium Alloy — never cell-tested. `right_only`: `B009`, `B010` — tests on materials with no properties on record. The `NaN` means **unknown / no record** — *not* zero, and *not* "failed."
</details>

---
### 🔨 Mini-Task B — Pick the Right Join (~3 min)

The materials team wants **every material we have mechanical data for, with cell-test results filled in where they exist**. Which `how=` gives that in one call? Assign to `props_with_bio` and check the shape.

<details><summary>Hint</summary>

"Every material we have mechanical data for" — which table has to keep *all* of its rows? Put it on the left.
</details>

<details><summary>Check yourself</summary>

`how="left"`, shape `(8, 9)`: all 8 materials in `properties`, 4 columns of `biocompatibility` added, `NaN` for `B005` and `B008`.
</details>

In [ ]:
# Your code here
props_with_bio = None

---
### 🔨 Task — Strong *and* Biocompatible? (~6 min)

**Real question:** which materials are both mechanically strong (`Tensile Strength (MPa)` **> 40**) and biocompatible (`Cell Viability (%)` **> 80**)?

1. **Join** `properties` and `biocompatibility` with the `how=` that keeps every material → `combined` (same choice as Mini-Task B)
2. **Filter** `combined` on *both* conditions → `strong_safe`. Which materials pass?
3. **Who never got judged?** Filter `combined` for rows with tensile strength > 40 **and a missing** `Cell Viability (%)` → `strong_untested`. (Hint: `.isna()`)

<details><summary>Hint</summary>

Combine conditions with `&` and wrap each in parentheses: `df[(df["a"] > 1) & (df["b"] > 2)]`. For missing values: `df["b"].isna()`.
</details>

<details><summary>Check yourself</summary>

`strong_safe`: **PLA** (50 MPa, 90 %) and **HA** (120 MPa, 95 %). `strong_untested`: **Titanium Alloy** (950 MPa!) and **Magnesium Alloy** (220 MPa) — strong, but never cell-tested. Step 2 silently dropped them (`NaN > 80` is `False`) — *not because they failed, but because nobody tested them.*
</details>

In [ ]:
# Your code here

---
## Part 3 — Does This Join Make Sense? (SLO 04B) · ~15 min

Every join so far *ran*. That's not the same as *making sense*. Before any `pd.merge()`, ask the three questions:

1. **Is the key really the same thing in both tables** — and unique where I think it is?
2. **Do the rows describe the same experiment** — same conditions, units, same duration?
3. **What will one row of the result be about?** Does that sentence even make sense?

Two things to check *after* merging, too: `indicator=True` (who found a partner?) and comparing `.shape` with the inputs (is the row count what you predicted?).

### Case 1 — A join that makes sense

`degradation`: one degradation test. `environment_conditions`: one environment. `Environment` in `degradation` is a foreign key to the primary key in `environment_conditions` — it's a lookup, so yes, the two rows describe the same thing. One row of the result will be about one degradation test, now carrying its environment's pH, temperature and composition.

Many tests share one environment ("many-to-one") — normal for a lookup table like this. `indicator=True` lets us check it: if the relationship is what we think, every row should come back `both`.

In [ ]:
deg_env = pd.merge(degradation, environment_conditions, on="Environment",
                   how="left", indicator=True)
print(deg_env.shape)
deg_env["_merge"].value_counts()

13 rows in, 13 rows out, and every row is `both` — a clean lookup. (There is exactly one `NaN` in the result: the `pH Range` for Air, which the CSV literally spells `N/A` and pandas reads as missing. Not a failed join — `_merge` proves it.)

### Case 2 — Joining on a column that isn't a key

Both tables have a `Condition` column, and pandas will happily join on it:

In [ ]:
by_condition = pd.merge(biocompatibility, biocompatibility_time, on="Condition")
by_condition.shape

Two 8-row tables produced **32** rows. `Condition` isn't an ID — it's a *category* (only two values), so every *in vitro* row gets paired with **every** *in vitro* row on the other side (4 × 4 = 16) and the same for *in vivo*. `B001`'s cell test is now attached to `B005`'s time series: rows that mean nothing.

This is exactly why the `.shape` check matters: nothing raised an error, but 32 rows out of two 8-row tables is the tell that `Condition` isn't behaving like a key at all.

---
### 🔨 Task — Is This Join Safe? (~5 min)

**Claim:** *"Attach material names to every degradation test."* → `degradation` + `properties` on `Material ID`, `how="inner"`.

**Run it, check the shape, and write a one-sentence verdict** — does it make sense, and what does one row of the result mean? (Ask the three questions from the top of Part 3, and compare the row count to the 13 tests `degradation` started with.)

<details><summary>Hint</summary>

To see who vanished, redo it with `how="left"` (or `"outer"`) and `indicator=True`.
</details>

<details><summary>Check yourself</summary>

**(7, 8) — right key, dangerous `how`.** Six of the 13 degradation tests (`B009`–`B014`) vanish because those materials have no properties on record. `Material ID` is a valid key here — the problem isn't the join, it's the choice of `how="inner"`. Use `how="left"` from `degradation` to keep all 13 and *report* the gap instead of silently dropping it.
</details>

In [ ]:
# Your code here


verdict = ""   # one sentence

---
## Careful with Joining · ~5 min

`inner` feels like the "safe" choice — no `NaN`s, everything filled in. But look at what an inner join of `properties` and `biocompatibility` did: it dropped **Titanium Alloy**, the strongest material in the lab (950 MPa). It didn't fail a cell test — it was never *given* one. An `inner`-only report would never even mention it.

Not a pandas bug — a **choice** (`how="inner"`) made once, early, that quietly shapes every number after it. And `NaN` from a join means **"no record"** — not zero, not "bad."

**A join is a claim that two rows describe the same thing.** pandas only checks that two values are equal. Before you merge:

1. Is the key really the same thing in both tables, and unique where I expect?
2. Same experiment — same conditions, units, durations?
3. What will one row of the result be about?

…and after: `indicator=True` and compare `.shape` to the inputs.

→ this week's reading traces the same kind of key mismatch through a full data journey — who collected it, who cleaned it, who's still missing.

---
## Coming Up

| Topic | What's next |
|---|---|
| This week's reading | Same kind of key mismatch, traced through a *data journey* — who collected it, who cleaned it, who's still missing |
| Practice | Melting, pivoting, joining together, on a new dataset |
| Week 5 | Clustering & Dimensionality Reduction — finding groups the data suggests, instead of ones we choose in advance |